In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict
from typing import Dict

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

combined_path = OUT_DIR / "v2_combined.csv"

# Let pandas detect the delimiter
data = pd.read_csv(combined_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data


In [ ]:
data.columns

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Identify and PRINT "unnecessary" ID-like columns and CODE-like
# columns from `data` without modifying it. Results are stored in variables for
# the next cell.
# -----------------------------------------------------------------------------
# Heuristics:
# • ID-like: names that are (case-insensitively) 'id', end with '_id'/'_ids',
#   start with 'id_', or contain '_id_' (e.g., 'student_id', 'cohort_ids').
# • CODE-like: names containing 'code' (e.g., 'degree_code', 'code_letters').
# • Always KEEP the primary key 'SINH_ID' (not flagged for removal).
# Outputs:
#   - unnecessary_id_cols   : list[str]
#   - unnecessary_code_cols : list[str]
# -----------------------------------------------------------------------------

cols = list(data.columns)

# 1) Identify ID-like columns (but exclude SINH_ID)
id_patterns = [
    r'^id$', r'^ids$',
    r'.*_id$', r'.*_ids$',
    r'^id_.*', r'^ids_.*',
    r'.*_id_.*', r'.*_ids_.*'
]
id_regex = re.compile('|'.join(id_patterns), flags=re.IGNORECASE)
unnecessary_id_cols = [c for c in cols if id_regex.search(c) and c.upper() != 'SINH_ID']

# 2) Identify CODE-like columns
code_regex = re.compile(r'code', flags=re.IGNORECASE)
unnecessary_code_cols = [c for c in cols if code_regex.search(c)]

# 3) Print results for review
print("ID-like columns to review (excluding 'SINH_ID'):")
for c in unnecessary_id_cols:
    print("-", c)
print(f"Total ID-like columns: {len(unnecessary_id_cols)}\n")

print("CODE-like columns to review:")
for c in unnecessary_code_cols:
    print("-", c)
print(f"Total CODE-like columns: {len(unnecessary_code_cols)}")


In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Drop columns listed in a user-defined mapping from `data`
# into a NEW DataFrame (`data_no_ids_codes`). 
# -----------------------------------------------------------------------------
# Instructions:
# 1) Define `cols_to_remove_map` yourself (e.g., {"colA": "id", "colB": "code"}).
# 2) This cell will:
#       • print the mapping,
#       • create `data_backup` (full copy),
#       • create `data_no_ids_codes` with those columns removed,
#       • report shapes.
# 3) It does NOT overwrite `data`. To adopt later:  data = data_no_ids_codes
# -----------------------------------------------------------------------------

# 1) Define your mapping here (column_name -> reason string like 'id' or 'code')
# 1) Define your list of columns to drop
cols_to_remove = [
    "sdo1_prev_distance_id",
    "sdo1_postal_distance_id",
    "sdo5_degree_degree_code_letters",
    "sdo5_degree_degree_code",
    "sdo5_degree_POSTCODE",
    "sdo1_previous_Previous_School_Postal_Code",
    "sdo1_prev_distance_postcode",
    "sdo1_postal_distance_postcode"
]

# 2) Print for confirmation
print("Columns to drop:")
for col in cols_to_remove:
    print("-", col)
print(f"\nTotal columns to dropped: {len(cols_to_remove)}")

# 3) Create backup and drop
data_backup = data.copy()
data_no_ids_codes = data.drop(columns=cols_to_remove, errors='ignore')

print("\nOriginal shape:", data.shape, "-> New shape:", data_no_ids_codes.shape)

# Optional adoption:
# data = data_no_ids_codes



In [ ]:
data = data_no_ids_codes

In [ ]:
# Ensure no two or more columns have the same values
identical_columns = []
columns = data.columns.tolist()

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1, col2 = columns[i], columns[j]
        if data[col1].equals(data[col2]):
            identical_columns.append((col1, col2))

if identical_columns:
    print("Identical columns found:")
    for col_pair in identical_columns:
        print(col_pair)
else:
    print("No identical columns found.")

In [ ]:
# Check for columns with only one unique value
single_value_cols = [col for col in data.columns if data[col].nunique() == 1]

print("Columns with only one unique value:", single_value_cols)


In [ ]:
# print the value 
print(data["has_results"].value_counts(dropna=False))

In [ ]:
# has_results has only one value, delete it

del data['has_results']
del data['has_characteristics']

In [ ]:
data.info()

In [ ]:
#pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)
#pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
#pd.set_option('display.expand_frame_repr', False)  


In [ ]:
# Print value counts for each column
for col in data.columns:
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")

In [ ]:
data.columns

## Create one drop-out date
Largely speaking a student either drops-out during the college year, or just simply does not re-register for the next year:

both are considered drop-outs. The drop-out dates for both types of drop-outs are administered in different data columns: 

if a student drops out during the year is, the date is registerd in `sdo5_degree_de_enrollment_date`, 

whereas if a student doesn't re-register at the end of the year the drop-out date is registered in `sdo5_degree_degree_end_date`. 

To get the most accurate drop-out date for all students these two fields are combined:

 (if) a student has a `sdo5_degree_de_enrollment_date` take this value, 
 
 (else) if take the `sdo5_degree_degree_end_date`. 

In [ ]:
# # Create one drop-out date
# Combine two drop-out date columns into one unified drop-out date column.
data['dropout_date'] = np.where(
    data['sdo5_degree_de_enrollment_date'].notna(),
    data['sdo5_degree_de_enrollment_date'],
    data['sdo5_degree_degree_end_date']
)

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Identify columns whose names contain 'date' or 'datum' (any case),
# print them, then convert those columns to pandas datetime.
# Notes:
# - Uses case-insensitive search for 'date' or 'datum'.
# - Converts IN PLACE on `data` using pd.to_datetime(errors='coerce', dayfirst=True).
#   (dayfirst=True is common for NL-style dates; adjust if needed.)
# -----------------------------------------------------------------------------

# 1) Find date-related columns
date_pattern = re.compile(r"(date|datum)", flags=re.IGNORECASE)
date_cols = [c for c in data.columns if date_pattern.search(c)]

# 2) Print for review
print("Detected date-related columns:")
for c in date_cols:
    print("-", c)
print(f"\nNumber of converted columns to date/time: {len(date_cols)}")

# 3) Convert to datetime (in place)
for c in date_cols:
    # Normalize obvious empties to NaN before parsing
    if data[c].dtype == object:
        data[c] = data[c].replace({"": np.nan, " ": np.nan, "NA": np.nan, "N/A": np.nan})
    data[c] = pd.to_datetime(data[c], errors="coerce", dayfirst=True)


In [ ]:
# 4) Display first 10 rows of date columns
print("\nSample values (head 10) from date-related columns:\n")
display(data[date_cols].head(10))


In [ ]:
# ------------------------------------------------------------------------------
# IN-PLACE FIX for corrupted degree dates using hidden YYYYMMDD after '0.'
# Overwrites:
#   - sdo5_degree_enrollment_date
#   - sdo5_degree_degree_start_date
#   - sdo5_degree_degree_end_date
# Also reparses the other date columns normally (dayfirst=True).
# ------------------------------------------------------------------------------

def repair_hidden_ymd(raw: pd.Series) -> pd.Series:
    """Extract YYYYMMDD hidden after '0.' (like 1970-...0.020180901) and parse."""
    s = raw.astype(str)

    # Grab 8–9 digits after '0.' anywhere in the string
    frac = s.str.extract(r'0\.(\d{8,9})')[0]

    # Normalize to 8-digit YYYYMMDD
    ymd = frac.str.lstrip('0').str[-8:]

    repaired = pd.to_datetime(ymd, format='%Y%m%d', errors='coerce')
    normal   = pd.to_datetime(s, errors='coerce')

    # Prefer repaired where it succeeded; else fall back to normal parse
    out = repaired.combine_first(normal)

    # Sanity filter: plausible years only
    with pd.option_context("mode.use_inf_as_na", True):
        year_ok = out.dt.year.between(1990, 2035, inclusive="both")
    out = out.where(year_ok, pd.NaT)

    # If anything still equals the Unix epoch date, drop it
    out = out.mask(out.dt.date.eq(pd.Timestamp("1970-01-01").date()), pd.NaT)

    return out

# Columns with the hidden-date bug to FIX IN PLACE
buggy_cols = [
    "sdo5_degree_enrollment_date",
    "sdo5_degree_degree_start_date",
    "sdo5_degree_degree_end_date",
    "dropout_date"
]

# Apply the repair in place (overwrite)
for col in buggy_cols:
    if col in data.columns:
        data[col] = repair_hidden_ymd(data[col])

# Other date-like columns: regular parse (they looked fine already)
other_date_cols = [
    "sdo1_previous_Final_Exam_Date",
    "sdo2_skc_SKC_DATUM",
    "sdo2_orientation_First_Event_Date",
    "sdo2_orientation_Last_Event_Date",
]
for c in other_date_cols:
    if c in data.columns:
        data[c] = pd.to_datetime(data[c], errors="coerce", dayfirst=True)

# ---- quick checks
print("Non-null shares after in-place fix:")
for c in buggy_cols:
    if c in data.columns:
        print(f"  {c}: {data[c].notna().mean():.1%}")

print("\nRanges after in-place fix:")
for c in buggy_cols:
    if c in data.columns:
        print(f"  {c}: {data[c].min()} → {data[c].max()}")

# Preview head(10)
cols_to_show = [c for c in (buggy_cols + other_date_cols) if c in data.columns]
display(data[cols_to_show].head(10))


In [ ]:
# Print value counts only for object (categorical) columns, excluding datetime columns and ID columns
# Columns to skip manually
skip_cols = ["SINH_ID"]

for col in data.select_dtypes(include=["object"]).columns:
    if col in skip_cols or col in data.select_dtypes(include=["datetime"]).columns:
        continue
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")


In [ ]:
# -----------------------------------------------------------------------------
# Purpose: For the next step, Print the object (categorical) columns that will be used
# in the next value-counts loop — excluding ID and datetime columns.
# -----------------------------------------------------------------------------

skip_cols = ["SINH_ID"]

dt_cols = set(data.select_dtypes(include=["datetime64[ns]"]).columns)
obj_cols_all = list(data.select_dtypes(include=["object"]).columns)
obj_cols = [c for c in obj_cols_all if c not in dt_cols and c not in skip_cols]

print("Object dtype columns (excluding datetime and skipped IDs):\n")
for col in obj_cols:
    print(col)
print(f"\nTotal object columns: {len(obj_cols)}")


----------------------- Handle categories in sdo5_degree_degree Column ---------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Clean and standardize `sdo5_degree_degree` program names for analysis.
# What this cell does:
#   • Applies exact mappings for specific categories:
#       - "B Education in Primary Schools (age 4 - 12) (day)"     → "Primary_Education_Day"
#       - "B Education in Primary Schools (age 4 - 12) (ALPO)"    → "Primary_Education_ALPO"
#       - "B Education in Primary Schools (age 4 - 12) (Regular)" → "Primary_Education_Regular"
#       - "B Arts Therapies (Music Therapy)"                      → "Music_Therapy"
#       - "B Arts Therapies (Drama Therapy)"                      → "Drama_Therapy"
#       - "B Arts Therapies (Art Therapy)"                        → "Art_Therapy"
#   • Removes leading degree prefixes: "B " and "Bachelor of ".
#   • Fixes known typo: "Chemics" → "Chemistry".
#   • Removes prepositions "and", "with", "in" (case-insensitive), commas, and ampersands.
#   • Normalizes whitespace, converts to Title Case, then replaces spaces with underscores.
#   • Preserves the standalone category "Teacher" exactly as-is.
#   • Converts NaN values to the label "Unknown".
# -----------------------------------------------------------------------------


col = "sdo5_degree_degree"

def clean_degree(value):
    if pd.isna(value):
        return "Unknown"
    
    v = str(value).strip()

    # ---- Exact mappings to keep as specified ----
    exact_map = {
        "B Education in Primary Schools (age 4 - 12) (day)": "Primary_Education_Day",
        "B Education in Primary Schools (age 4 - 12) (ALPO)": "Primary_Education_ALPO",
        "B Education in Primary Schools (age 4 - 12) (Regular)": "Primary_Education_Regular",
        "B Arts Therapies (Music Therapy)": "Music_Therapy",
        "B Arts Therapies (Drama Therapy)": "Drama_Therapy",
        "B Arts Therapies (Art Therapy)": "Art_Therapy",
        # Optional: make "Bachelor of Law" explicit instead of "Law"
        # "Bachelor of Law": "Law_Bachelor",
    }
    if v in exact_map:
        return exact_map[v]

    # ---- Remove prefixes ----
    v = re.sub(r"^\s*B\s+", "", v)
    v = re.sub(r"^\s*Bachelor of\s+", "", v, flags=re.IGNORECASE)

    # ---- Fix known inconsistencies ----
    v = v.replace("Chemics", "Chemistry")

    # ---- Remove prepositions and punctuation ----
    v = v.replace("&", " ")
    v = re.sub(r"\b(?:and|with|in)\b", "", v, flags=re.IGNORECASE)
    v = v.replace(",", " ")

    # ---- Normalize whitespace and casing ----
    v = re.sub(r"\s+", " ", v).strip()
    v = v.title()

    # ---- Replace spaces with underscores ----
    v = v.replace(" ", "_")

    # ---- NEW: collapse multiple underscores and trim edges ----
    v = re.sub(r"_+", "_", v).strip("_")

    return v

# Apply cleaning directly to the original column
data[col] = data[col].apply(clean_degree)

# Optional check
print(data[col].value_counts())

----------------------- Handle categories in sdo1_characteristics_gender Column ---------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: In `sdo1_characteristics_gender`, exclude all students with 'O'/'o'.
# For the remaining students, replace NaN with 'Unknown' without altering other
# valid values (e.g., 'M', 'V').
# -----------------------------------------------------------------------------

col = "sdo1_characteristics_gender"

# Exclude rows where gender is 'O' or 'o'
data = data[~data[col].isin(['O', 'o'])]

# If the column is categorical, ensure 'Unknown' exists as a category
if pd.api.types.is_categorical_dtype(data[col]):
    data[col] = data[col].cat.add_categories(["Unknown"])

# Replace NaN with 'Unknown'
data[col] = data[col].fillna("Unknown")

print(data['sdo1_characteristics_gender'].value_counts())


------------------- Handle categories in sdo1_previous_Previous_Education_Type Column ---------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: In `sdo1_previous_Previous_Education_Type`, replace NaN and literal
# '$' with 'Unknown
# -----------------------------------------------------------------------------


col = "sdo1_previous_Previous_Education_Type"

# If categorical, ensure 'Unknown' is an allowed category
if pd.api.types.is_categorical_dtype(data[col]):
    data[col] = data[col].cat.add_categories(["Unknown"])

# Replace and fill
data[col] = data[col].replace({"$": "Unknown"}).fillna("Unknown")

# Print value counts after replacement
print(data[col].value_counts())


-------------------------------------- Handle categories in sdo2_ADVIES_DEF Column -------------------------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Replace NaN with 'Unknown' and add underscores between words
# in the categories of `sdo2_skc_ADVIES_DEF.
# -----------------------------------------------------------------------------

col = "sdo2_skc_ADVIES_DEF"

# 1) Add underscores between words for non-missing values
data[col] = data[col].apply(lambda x: "_".join(str(x).split()) if pd.notna(x) else x)

# 2) Replace NaN with 'Niet_deelgenomen = not participated'
data[col] = data[col].fillna("Niet_deelgenomen")
print(data[col].value_counts())

-------------------------------------- Handle categories in sdo2_orientation_Event_Types_Attended Column -------------------------------------------

In [ ]:
# Expand truncated output to print all the value counts
s = data['sdo2_orientation_Event_Types_Attended'].value_counts(dropna=False)

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    print(s)          # or: display(s) in Jupyter


In [ ]:
# =============================================================================
# FUNCTION: Cleaning_sdo2_orientation_Event_Types_Attended
# -----------------------------------------------------------------------------
# PURPOSE
# Clean & categorize the column: sdo2_orientation_Event_Types_Attended
#
# Steps:
# 1) Fill missing with 'absent', lowercase, and strip spaces.
# 2) Normalize English/Dutch synonyms:
#       - 'open day' / 'orientation day' -> 'open dag'
#       - 'guided tour'                  -> 'rondleiding'
#       - 'online open evening/night/avond' -> 'open avond'
#       - 'kennismaken op locatie' / 'international hour' -> 'kennismaken'
# 3) Split comma-separated items, trim, de-duplicate.
# 4) Reassemble in canonical order:
#       ['open dag','open avond','online opleidingspresentatie',
#        'proefstuderen','meeloopdag','rondleiding','kennismaken','vragenuur']
# 5) Overwrite the column with its final category:
#       - Absent / Multiple_Types / Open_Day_Related / Trial_or_Experience_Day /
#         Campus_Tour / Meet_and_Greet / Other
# 6) Print category counts.
# =============================================================================

def Cleaning_sdo2_orientation_Event_Types_Attended(data: pd.DataFrame) -> pd.DataFrame:
    col_in = "sdo2_orientation_Event_Types_Attended"

    # 1) Fill missing, lowercase, strip
    ser = data[col_in].fillna("absent").astype(str).str.strip().str.lower()

    # 2) Normalize synonyms
    synonym_map = {
        "open day": "open dag",
        "orientation day": "open dag",
        "guided tour": "rondleiding",
        "online open evening": "open avond",
        "online open night": "open avond",
        "online open avond": "open avond",
        "kennismaken op locatie": "kennismaken",
        "international hour": "kennismaken",
    }
    for src, dst in synonym_map.items():
        ser = ser.str.replace(src, dst, regex=False)

    # 3–4) Normalize & reorder
    order = [
        "open dag",
        "open avond",
        "online opleidingspresentatie",
        "proefstuderen",
        "meeloopdag",
        "rondleiding",
        "kennismaken",
        "vragenuur",
    ]
    rank = {name: i for i, name in enumerate(order)}

    def normalize_row(val: str) -> str:
        if val in ("", "absent", "none", "nan"):
            return "absent"
        parts = [p.strip() for p in val.split(",")]
        uniq = sorted(set([p for p in parts if p]), key=lambda x: rank.get(x, 999))
        return ", ".join(uniq) if uniq else "absent"

    ser = ser.apply(normalize_row)

    # 5) Categorize — overwrite the same column
    def categorize(clean_val: str) -> str:
        if clean_val == "absent":
            return "Absent"
        items = [p.strip() for p in clean_val.split(",") if p.strip()]
        if len(items) > 1:
            return "Multiple_Types"
        item = items[0]
        if item in {"open dag", "open avond", "online opleidingspresentatie"}:
            return "Open_Day_Related"
        if item in {"proefstuderen", "meeloopdag"}:
            return "Trial_or_Experience_Day"
        if item == "rondleiding":
            return "Campus_Tour"
        if item == "kennismaken":
            return "Meet_and_Greet"
        return "Other"

    data[col_in] = ser.apply(categorize)

    # 6) Summary
    print("\nFinal Category Counts (sdo2_orientation_Event_Types_Attended):")
    print(data[col_in].value_counts(dropna=False))
    return data

# Example usage:
data = Cleaning_sdo2_orientation_Event_Types_Attended(data)


In [ ]:
# Confirm NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False).head(25)
print(nan_sums)

In [ ]:
data.to_csv(f"{OUT_DIR}/cleaned_original.csv", index=False)
print(f"✅ Saved to: {OUT_DIR}/cleaned_original.csv")